# 第14章 深層学習の基礎と生成AIの活用

『Python機械学習スタートブック』のコードをGoogle Colabで実行するためのノートブックです。
コードは書籍のリスト番号順に並んでいます。上から順に実行してください。

- 解説（Web教材）: https://ml.kano.ac/chapters/14/
- 演習の解答: https://ml.kano.ac/solutions/14/

## 深層学習の実践

### TensorFlowとKerasの準備

**リスト 14.1**　TensorFlowのインストール

In [ ]:
# Google Colabでは不要（プリインストール済み）
# ローカル環境の場合のみ実行
# !pip install tensorflow

**リスト 14.2**　TensorFlowのバージョン確認

In [ ]:
import tensorflow as tf
print(tf.__version__)

### MNISTデータセットによる手書き数字分類

**リスト 14.3**　MNISTデータの読み込みと前処理

In [ ]:
from tensorflow.keras.datasets import mnist

# データの読み込み（MNISTは11章でも使用）
(X_train, y_train), (X_test, y_test) = mnist.load_data()

# ピクセル値を0〜1に正規化
X_train = X_train / 255.0
X_test = X_test / 255.0

# 28x28の画像を784次元のベクトルに変換（Flatten）
X_train_flat = X_train.reshape(-1, 784)
X_test_flat = X_test.reshape(-1, 784)

print(f"変換後の訓練データ: {X_train_flat.shape}")

### Kerasによるモデルの構築

**リスト 14.4**　全結合ニューラルネットワークの構築

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Input

# モデルの構築
model = Sequential([
    Input(shape=(784,)),              # 入力の形状を指定
    Dense(128, activation="relu"),    # 隠れ層1
    Dense(64, activation="relu"),     # 隠れ層2
    Dense(10, activation="softmax")   # 出力層（10クラス）
])

# モデルの概要を表示
model.summary()

### モデルのコンパイルと学習

**リスト 14.5**　モデルのコンパイルと学習

In [ ]:
# モデルのコンパイル
model.compile(
    optimizer="adam",                        # オプティマイザ
    loss="sparse_categorical_crossentropy",  # 損失関数
    metrics=["accuracy"]                     # 評価指標
)

# モデルの学習
history = model.fit(
    X_train_flat, y_train,
    epochs=10,           # 全データを10回繰り返し学習
    batch_size=32,       # 32サンプルずつまとめて計算
    validation_split=0.2 # 訓練データの20%を検証用に使用
)

### モデルの評価

**リスト 14.6**　テストデータによるモデルの評価

In [ ]:
# テストデータで評価
test_loss, test_accuracy = model.evaluate(X_test_flat, y_test)
print(f"テスト精度: {test_accuracy:.4f}")

**リスト 14.7**　学習曲線の描画

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# 精度の推移
axes[0].plot(history.history["accuracy"], label="訓練")
axes[0].plot(history.history["val_accuracy"], label="検証")
axes[0].set_title("精度の推移")
axes[0].set_xlabel("エポック")
axes[0].set_ylabel("精度")
axes[0].legend()

# 損失の推移
axes[1].plot(history.history["loss"], label="訓練")
axes[1].plot(history.history["val_loss"], label="検証")
axes[1].set_title("損失の推移")
axes[1].set_xlabel("エポック")
axes[1].set_ylabel("損失")
axes[1].legend()

plt.tight_layout()
plt.show()

## 画像における深層学習：畳み込みニューラルネットワーク（CNN）

### KerasによるCNNの実装

**リスト 14.8**　CNN入力形状へのデータ変換

In [ ]:
from tensorflow.keras.datasets import mnist

# データの再読み込みと前処理
(X_train, y_train), (X_test, y_test) = mnist.load_data()
X_train = X_train / 255.0
X_test = X_test / 255.0

# CNNの入力形状に変換: (サンプル数, 高さ, 幅, チャンネル数)
X_train_cnn = X_train.reshape(-1, 28, 28, 1)
X_test_cnn = X_test.reshape(-1, 28, 28, 1)

print(f"CNN入力の形状: {X_train_cnn.shape}")

**リスト 14.9**　CNNモデルの構築

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    Input, Conv2D, MaxPooling2D, Flatten, Dense
)

# CNNモデルの構築
cnn_model = Sequential([
    Input(shape=(28, 28, 1)),
    Conv2D(32, (3, 3), activation="relu"),
    MaxPooling2D((2, 2)),
    Conv2D(64, (3, 3), activation="relu"),
    MaxPooling2D((2, 2)),
    Flatten(),
    Dense(64, activation="relu"),
    Dense(10, activation="softmax")
])

cnn_model.summary()

### CNNモデルの学習と評価

**リスト 14.10**　CNNモデルのコンパイルと学習

In [ ]:
# コンパイルと学習
cnn_model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

cnn_history = cnn_model.fit(
    X_train_cnn, y_train,
    epochs=5,
    batch_size=64,
    validation_split=0.2
)

**リスト 14.11**　CNNモデルの評価

In [ ]:
# テストデータで評価
test_loss, test_accuracy = cnn_model.evaluate(X_test_cnn, y_test)
print(f"CNNのテスト精度: {test_accuracy:.4f}")

### カラー画像の分類：CIFAR-10

**リスト 14.12**　CIFAR-10データの読み込み

In [ ]:
from tensorflow.keras.datasets import cifar10

# データの読み込み（初回はダウンロードに時間がかかる）
(X_train, y_train), (X_test, y_test) = cifar10.load_data()

print(f"訓練データ: {X_train.shape}, ラベル: {y_train.shape}")
print(f"テストデータ: {X_test.shape}, ラベル: {y_test.shape}")

**リスト 14.13**　CIFAR-10用CNNの構築と学習

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    Input, Conv2D, MaxPooling2D, Flatten, Dense
)

# ピクセル値を0〜1に正規化
X_train = X_train / 255.0
X_test = X_test / 255.0

# MNISTのCNNと同じ構造（入力形状のみ変更）
cifar_model = Sequential([
    Input(shape=(32, 32, 3)),
    Conv2D(32, (3, 3), activation="relu"),
    MaxPooling2D((2, 2)),
    Conv2D(64, (3, 3), activation="relu"),
    MaxPooling2D((2, 2)),
    Flatten(),
    Dense(64, activation="relu"),
    Dense(10, activation="softmax")
])

cifar_model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

cifar_history = cifar_model.fit(
    X_train, y_train,
    epochs=10,
    batch_size=64,
    validation_split=0.2
)

**リスト 14.14**　CIFAR-10モデルの評価

In [ ]:
# テストデータで評価
test_loss, test_accuracy = cifar_model.evaluate(X_test, y_test)
print(f"CIFAR-10のテスト精度: {test_accuracy:.4f}")

## テキストにおける深層学習：Transformer

### Hugging Face transformersライブラリ

**リスト 14.15**　`transformers`のインストール

In [ ]:
# Google Colabでは不要（プリインストール済み）
# ローカル環境の場合のみ実行
# !pip install transformers

### パイプラインによる感情分析

**リスト 14.16**　パイプラインによる感情分析

In [ ]:
from transformers import pipeline

# 感情分析パイプラインの作成
classifier = pipeline("sentiment-analysis")

# 英語テキストの感情分析
texts = [
    "I love this movie! It was absolutely wonderful.",
    "This product is terrible. I want a refund.",
    "The weather is okay today."
]
results = classifier(texts)

for text, result in zip(texts, results):
    print(f"{text}: {result['label']} ({result['score']:.4f})")

### 日本語テキストの感情分析

**リスト 14.17**　日本語テキストの感情分析

In [ ]:
from transformers import pipeline

# 日本語感情分析モデルの利用
classifier_ja = pipeline(
    "sentiment-analysis",
    model="lxyuan/distilbert-base-multilingual-cased-sentiments-student"
)

# 日本語テキストの感情分析
texts_ja = [
    "この映画は最高でした！感動しました。",
    "サービスが悪くて非常に不満です。",
    "今日のランチは普通でした。"
]

results_ja = classifier_ja(texts_ja)

for text, result in zip(texts_ja, results_ja):
    print(f"{text}")
    print(f"  → {result['label']} ({result['score']:.4f})")
    print()

### テキスト分類の実践

**リスト 14.18**　ゼロショット分類の実行

In [ ]:
from transformers import pipeline

# ゼロショット分類
zero_shot = pipeline(
    "zero-shot-classification",
    model="facebook/bart-large-mnli"
)

text = "Apple announced a new iPhone model with improved camera."
candidate_labels = ["technology", "sports", "politics", "entertainment"]

result = zero_shot(text, candidate_labels)

for label, score in zip(result["labels"], result["scores"]):
    print(f"  {label}: {score:.4f}")

## 生成AIの活用

### APIを使ったテキスト生成

**リスト 14.19**　`openai`ライブラリのインストール

In [ ]:
# Google Colabでは不要（プリインストール済み）
# ローカル環境の場合のみ実行
# !pip install openai

**リスト 14.20**　OpenAI APIによるテキスト生成

In [ ]:
from openai import OpenAI
import getpass
import os

# APIキーを画面に表示せず入力し、環境変数に設定
if "OPENAI_API_KEY" not in os.environ:
    os.environ["OPENAI_API_KEY"] = getpass.getpass("OpenAI API key: ")

# 環境変数 OPENAI_API_KEY を自動的に読み込む
client = OpenAI()

# テキスト生成
response = client.responses.create(
    model="gpt-5-mini",
    instructions="あなたは親切なアシスタントです。",
    input="機械学習とは何かを3行で説明してください。"
)

print(response.output_text)

### プロンプトエンジニアリング

**リスト 14.21**　プロンプトの書き方による出力の比較

In [ ]:
# 曖昧なプロンプト
response1 = client.responses.create(
    model="gpt-5-mini",
    input="リスト内包表記について教えてください。"
)
print("【曖昧なプロンプト】")
print(response1.output_text[:200])
print("...")

# 具体的なプロンプト
response2 = client.responses.create(
    model="gpt-5-mini",
    instructions="あなたはプログラミング講師です。初心者向けに説明してください。",
    input="Pythonのリスト内包表記について、具体例を1つ含めて"
          "3行以内で説明してください。"
)
print("\n【具体的なプロンプト】")
print(response2.output_text)

## 演習問題

### 演習 14-1: 活性化関数の理解

以下のコードを実行し、ReLU、シグモイド、ソフトマックスの各活性化関数の出力を確認してください。それぞれの関数がどのような変換を行っているか説明してください。

In [ ]:
import numpy as np

x = np.array([-2.0, -1.0, 0.0, 1.0, 2.0])

# ReLU
relu = np.maximum(0, x)
print(f"ReLU: {relu}")

# シグモイド
sigmoid = 1 / (1 + np.exp(-x))
print(f"Sigmoid: {sigmoid}")

# ソフトマックス
softmax = np.exp(x) / np.sum(np.exp(x))
print(f"Softmax: {softmax}")
print(f"Softmax合計: {np.sum(softmax)}")

[解答例を見る](https://ml.kano.ac/solutions/14/#solution-14-1)

### 演習 14-2: Fashion-MNISTでのCNN

以下のコードを使って、Fashion-MNIST データセットを読み込み、CNN モデルで分類を行ってください。テスト精度はいくつになるか確認してください。また、テストデータの先頭数枚について、予測結果を `label_names` のラベル名で表示してみましょう。

In [ ]:
from tensorflow.keras.datasets import fashion_mnist

# データの読み込み
(X_train, y_train), (X_test, y_test) = fashion_mnist.load_data()
X_train = X_train / 255.0
X_test = X_test / 255.0
X_train = X_train.reshape(-1, 28, 28, 1)
X_test = X_test.reshape(-1, 28, 28, 1)

# ラベル名
label_names = ["Tシャツ", "ズボン", "セーター", "ドレス", "コート",
               "サンダル", "シャツ", "スニーカー", "バッグ", "ブーツ"]

# 本節のCNNモデルと同じ構造でモデルを構築し、学習・評価してください

[解答例を見る](https://ml.kano.ac/solutions/14/#solution-14-2)

### 演習 14-3: Hugging Faceパイプラインの活用

Hugging Face の `pipeline` を使って、以下のタスクを実行してください。

1. 以下の 3 つの日本語テキストに対して感情分析を行い、結果を確認してください。
    - 「この本はとても勉強になりました」
    - 「電車が遅延して最悪でした」
    - 「今日は特に何もありませんでした」
2. `pipeline("text-generation")` を使って、英語のテキスト生成を試してください。プロンプトとして `"Machine learning is"` を入力し、生成されるテキストを確認してください。

[解答例を見る](https://ml.kano.ac/solutions/14/#solution-14-3)

### 演習 14-4: ニューラルネットワークの構造変更

本章の MNIST 全結合ネットワークの構造を以下のように変更し、テスト精度がどのように変化するか比較してください。

1. 隠れ層を 1 層(Dense(128) のみ)にした場合
2. 隠れ層を 3 層(Dense(256), Dense(128), Dense(64))にした場合
3. 隠れ層のニューロン数を少なくした場合(Dense(32), Dense(16))

結果を表にまとめ、ネットワーク構造と精度の関係を考察してください。

[解答例を見る](https://ml.kano.ac/solutions/14/#solution-14-4)

### 演習 14-5: プロンプトエンジニアリングの実践

以下の課題について、異なるプロンプトを 3 パターン以上作成し、LLM（ChatGPT や Claude など）に入力して、出力の違いを比較してください。

- 課題：「機械学習でよく使われる分類アルゴリズムを比較する表を作成してください」

どのようなプロンプトを書いたときに最も有用な出力が得られたか、その理由とともに考察してください。

[解答例を見る](https://ml.kano.ac/solutions/14/#solution-14-5)